# LLM Adaptation Study — LoRA fine-tuning vs in-context learning

Runs on a **free Colab T4**.

Before you start: **Runtime → Change runtime type → T4 GPU**.

The code lives in the repository, not in this notebook. This file clones it,
installs what it needs, and runs it. That way the logic is version controlled
and tested, and the notebook stays something you can re-run from scratch.

In [ ]:
!nvidia-smi
import torch
print("CUDA:", torch.cuda.is_available())
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("bfloat16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

A T4 reports `bfloat16 supported: False`. That is expected and the code
checks for it — training in bf16 on a Turing card either fails or silently
crawls, which is the most common reason a notebook that worked on an A100 does
not work here.

In [ ]:
!git clone https://github.com/MihirParekh14/llm-adaptation-study.git
%cd llm-adaptation-study
!pip install -q -r requirements.txt

## Hugging Face token

Llama 3.2 and Gemma 2 are gated. Add your token to Colab's secrets:
the key icon in the left sidebar → **Add new secret** → name it `HF_TOKEN`,
paste the token, and turn on **Notebook access**.

Mistral and Qwen need no token, so skip this cell if you only want those.

In [ ]:
from google.colab import userdata
import os
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import login
login(os.environ["HF_TOKEN"])
print("signed in")

## Check the tests pass here too

Thirty seconds. If the parser behaves differently in this environment, you
want to know now rather than after an hour of GPU time.

In [ ]:
!pip install -q pytest
!python -m pytest -q

## A quick run first

One ungated model, a small evaluation set. About ten minutes. Confirms the
whole path works before you spend an hour on it.

In [ ]:
!python -m src.run --models qwen-2.5-3b --eval-examples 100 --train-examples 500

## The full run

All four models, three methods each. Expect roughly 60–90 minutes on a T4.

Colab disconnects idle sessions, so keep the tab open. If it dies partway, the
results printed so far are still in the output above.

In [ ]:
!python -m src.run \
    --models llama-3.2-3b gemma-2-2b mistral-7b qwen-2.5-3b \
    --eval-examples 400 \
    --train-examples 2000

## If a gated model fails

Run the rest rather than losing the session:

```
!python -m src.run --ungated-only
```

In [ ]:
# Save the results back to Drive so they survive the session ending.
from google.colab import drive
drive.mount('/content/drive')
!cp results/results.json /content/drive/MyDrive/llm_adaptation_results.json
print("saved")